## AI-Powered Educational Chatbot

## STEP1 : Importing Library and Data Loading 

In [1]:
import numpy as np
import pandas as pd


In [2]:
df = pd.read_csv('knowledge_datascience.csv')


In [4]:
df1 = df.copy()

In [5]:
df1.head()

,question,answer
0,What is Artificial Intelligence?,Artificial Intelligence (AI) is the field of c...
1,What is an AI agent?,An AI agent is a system that perceives its env...
2,What is supervised AI?,Supervised AI usually refers to AI systems tra...
3,What is weak AI?,"Weak AI, or narrow AI, is designed to perform ..."
4,What is artificial general intelligence?,Artificial General Intelligence (AGI) is a hyp...


## STEP2 : Data Cleaning

In [6]:
df1.shape ## It is total shape of whole dataset

(13312, 2)

In [7]:
df1.isna().sum() ## It is predicting , there is no null value in this whole dataset.

question    0
answer      0
dtype: int64

In [8]:
df1.info() ## There is information about whole dataset.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13312 entries, 0 to 13311
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   question  13312 non-null  object
 1   answer    13312 non-null  object
dtypes: object(2)
memory usage: 208.1+ KB


In [9]:
df1.duplicated().sum()  ## There is no duplicate value

0

In [10]:
import re

# Create a copy of the original dataset
df_clean = df1.copy()

# Convert question and answer to string
df_clean['question'] = df_clean['question'].astype(str)
df_clean['answer'] = df_clean['answer'].astype(str)

# Remove leading and trailing spaces
df_clean['question'] = df_clean['question'].str.strip()
df_clean['answer'] = df_clean['answer'].str.strip()

# Remove HTML tags
df_clean['question'] = df_clean['question'].str.replace(
    r'<[^>]+>', '', regex=True
)

df_clean['answer'] = df_clean['answer'].str.replace(
    r'<[^>]+>', '', regex=True
)

# Replace multiple spaces/newlines with a single space
df_clean['question'] = df_clean['question'].str.replace(
    r'\s+', ' ', regex=True
)

df_clean['answer'] = df_clean['answer'].str.replace(
    r'\s+', ' ', regex=True
)

# Remove duplicate question-answer pairs
df_clean = df_clean.drop_duplicates(
    subset=['question', 'answer']
).reset_index(drop=True)

# Display cleaned data
df_clean.head()

,question,answer
0,What is Artificial Intelligence?,Artificial Intelligence (AI) is the field of c...
1,What is an AI agent?,An AI agent is a system that perceives its env...
2,What is supervised AI?,Supervised AI usually refers to AI systems tra...
3,What is weak AI?,"Weak AI, or narrow AI, is designed to perform ..."
4,What is artificial general intelligence?,Artificial General Intelligence (AGI) is a hyp...


## STEP3 : Data Embedding and Model Deployment(Transformer)

In [11]:
!pip install sentence-transformers



[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:

from sentence_transformers import SentenceTransformer

# Load pretrained Sentence Transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for all questions
question_embeddings = model.encode(
    df_clean['question'].tolist(),
    normalize_embeddings=True,
    show_progress_bar=True
)

# Check embedding shape
print("Embedding Shape:", question_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/416 [00:00<?, ?it/s]

Embedding Shape: (13312, 384)


## STEP5 : Testing Model and Cosine Similarity

In [81]:
# ============================================================
# TOPIC DETECTION
# ============================================================

TOPIC_KEYWORDS = {

    "Python": [
        "python",
        "python programming",
        "python language",
        "python code",
        "python programming language"
    ],

    "SQL": [
        "sql",
        "mysql",
        "database query",
        "sql query",
        "sql database"
    ],

    "Machine Learning": [
        "machine learning",
        "ml",
        "supervised learning",
        "unsupervised learning",
        "classification",
        "regression"
    ],

    "Deep Learning": [
        "deep learning",
        "neural network",
        "cnn",
        "rnn",
        "lstm",
        "deep neural network"
    ],

    "Data Science": [
        "data science",
        "data scientist",
        "data analysis",
        "data analytics"
    ],

    "Statistics": [
        "statistics",
        "statistical",
        "mean",
        "median",
        "standard deviation",
        "variance",
        "probability"
    ],

    "NLP": [
        "nlp",
        "natural language processing",
        "text processing",
        "tokenization",
        "stemming",
        "lemmatization"
    ],

    "Computer Vision": [
        "computer vision",
        "image processing",
        "image classification",
        "object detection",
        "image segmentation",
        "opencv"
    ],

    "Generative AI": [
        "generative ai",
        "gen ai",
        "chatgpt",
        "large language model",
        "llm",
        "text generation"
    ],

    "Artificial Intelligence": [
        "artificial intelligence",
        "ai",
        "intelligent system"
    ]
}


def detect_topic(question):

    question = question.lower().strip()

    # Check longer phrases first
    all_keywords = []

    for topic, keywords in TOPIC_KEYWORDS.items():

        for keyword in keywords:
            all_keywords.append(
                (keyword, topic)
            )

    all_keywords.sort(
        key=lambda x: len(x[0]),
        reverse=True
    )

    for keyword, topic in all_keywords:

        # Whole-word matching
        pattern = r"\b" + re.escape(keyword) + r"\b"

        if re.search(pattern, question):

            return topic

    return None

In [82]:
# ============================================================
# FOLLOW-UP QUESTION DETECTION
# ============================================================

FOLLOW_UP_PATTERNS = [

    r"\bit\b",
    r"\bits\b",
    r"\bit's\b",
    r"\bthey\b",
    r"\bthem\b",
    r"\btheir\b",
    r"\bthis\b",
    r"\bthat\b",
    r"\bthese\b",
    r"\bthose\b",

    r"what is its",
    r"what are its",
    r"what about",
    r"how about",
    r"why is it",
    r"why are they",
    r"how does it",
    r"how do they",
    r"explain it",
    r"explain further",
    r"tell me more",
    r"give me more",
    r"give an example",
    r"examples of it",
    r"advantages of it",
    r"disadvantages of it"
]


def is_follow_up(question):

    question = question.lower().strip()

    # Check follow-up patterns
    for pattern in FOLLOW_UP_PATTERNS:

        if re.search(pattern, question):

            return True

    # Very short questions are often follow-ups
    words = question.split()

    if len(words) <= 4:

        short_patterns = [
            "why",
            "how",
            "what about",
            "and",
            "then",
            "examples",
            "example",
            "types",
            "type",
            "advantages",
            "disadvantages",
            "uses",
            "applications"
        ]

        for pattern in short_patterns:

            if question.startswith(pattern):

                return True

    return False

In [83]:
# ============================================================
# FOLLOW-UP INTENT
# ============================================================

def detect_intent(question):

    question = question.lower().strip()

    if (
        "type" in question
        or "types" in question
        or "kind" in question
        or "kinds" in question
    ):
        return "types"

    if (
        "example" in question
        or "examples" in question
    ):
        return "examples"

    if (
        "advantage" in question
        or "advantages" in question
    ):
        return "advantages"

    if (
        "disadvantage" in question
        or "disadvantages" in question
    ):
        return "disadvantages"

    if (
        "use" in question
        or "uses" in question
        or "application" in question
        or "applications" in question
    ):
        return "uses"

    if (
        "definition" in question
        or "define" in question
        or "what is" in question
    ):
        return "definition"

    if (
        "how" in question
    ):
        return "how"

    if (
        "why" in question
    ):
        return "why"

    return "general"

In [84]:
# ============================================================
# CONTEXT MANAGER
# ============================================================

class ContextAnalyzer:

    def __init__(self):

        self.active_topic = None


    def analyze(self, question):

        question = question.strip()

        # Detect explicit topic
        detected_topic = detect_topic(question)

        # ----------------------------------------------------
        # CASE 1: New topic explicitly mentioned
        # ----------------------------------------------------

        if detected_topic is not None:

            self.active_topic = detected_topic

            return {
                "topic": self.active_topic,
                "is_follow_up": False,
                "intent": detect_intent(question),
                "query": question
            }

        # ----------------------------------------------------
        # CASE 2: Follow-up question
        # ----------------------------------------------------

        if is_follow_up(question):

            intent = detect_intent(question)

            if self.active_topic is not None:

                # Create a focused contextual query
                if intent == "types":

                    contextual_query = (
                        self.active_topic
                        + " data types "
                        + question
                    )

                elif intent == "examples":

                    contextual_query = (
                        self.active_topic
                        + " examples "
                        + question
                    )

                elif intent == "advantages":

                    contextual_query = (
                        self.active_topic
                        + " advantages "
                        + question
                    )

                elif intent == "disadvantages":

                    contextual_query = (
                        self.active_topic
                        + " disadvantages "
                        + question
                    )

                elif intent == "uses":

                    contextual_query = (
                        self.active_topic
                        + " uses applications "
                        + question
                    )

                else:

                    contextual_query = (
                        self.active_topic
                        + " "
                        + question
                    )

                return {
                    "topic": self.active_topic,
                    "is_follow_up": True,
                    "intent": intent,
                    "query": contextual_query
                }

        # ----------------------------------------------------
        # CASE 3: No topic found
        # ----------------------------------------------------

        return {
            "topic": self.active_topic,
            "is_follow_up": False,
            "intent": detect_intent(question),
            "query": question
        }


# Create one context analyzer
context_analyzer = ContextAnalyzer()

In [85]:
# ============================================================
# CONTEXT-AWARE GET ANSWER
# ============================================================

def get_answer(user_question, threshold=0.55):

    user_question = user_question.strip()

    # --------------------------------------------------------
    # EMPTY QUESTION
    # --------------------------------------------------------

    if not user_question:

        return {
            "answer": "Please enter a question.",
            "matched_question": None,
            "score": 0.0,
            "matched": False,
            "topic": None,
            "is_follow_up": False,
            "intent": None,
            "search_query": ""
        }


    # --------------------------------------------------------
    # ANALYZE CONTEXT
    # --------------------------------------------------------

    context = context_analyzer.analyze(
        user_question
    )

    topic = context["topic"]

    is_follow_up_question = (
        context["is_follow_up"]
    )

    intent = context["intent"]

    search_query = context["query"]


    # --------------------------------------------------------
    # CREATE QUERY EMBEDDING
    # --------------------------------------------------------

    query_embedding = model.encode(
        search_query,
        normalize_embeddings=True
    )


    # --------------------------------------------------------
    # INITIAL SIMILARITY
    # --------------------------------------------------------

    similarity_scores = cosine_similarity(
        [query_embedding],
        question_embeddings
    )[0]


    # --------------------------------------------------------
    # TOPIC FILTERING
    #
    # If we know the active topic, search mainly
    # inside that topic.
    # --------------------------------------------------------

    candidate_indices = np.arange(
        len(df)
    )

    if topic is not None:

        topic_indices = []

        for i, question in enumerate(
            df["question"]
        ):

            detected = detect_topic(
                question
            )

            if detected == topic:

                topic_indices.append(i)

        if len(topic_indices) > 0:

            candidate_indices = np.array(
                topic_indices
            )


    # --------------------------------------------------------
    # GET CANDIDATE SCORES
    # --------------------------------------------------------

    candidate_scores = (
        similarity_scores[candidate_indices]
    )


    # --------------------------------------------------------
    # INTENT BOOST
    #
    # Helps "What is its type?"
    # find "Python data types"
    # instead of another Python question.
    # --------------------------------------------------------

    boosted_scores = (
        candidate_scores.copy()
    )


    for position, index in enumerate(
        candidate_indices
    ):

        kb_question = (
            df.iloc[index]["question"]
            .lower()
        )

        # Types
        if intent == "types":

            if (
                "type" in kb_question
                or "types" in kb_question
                or "data type" in kb_question
                or "data types" in kb_question
            ):

                boosted_scores[position] += 0.10


        # Examples
        elif intent == "examples":

            if (
                "example" in kb_question
                or "examples" in kb_question
            ):

                boosted_scores[position] += 0.08


        # Advantages
        elif intent == "advantages":

            if (
                "advantage" in kb_question
                or "advantages" in kb_question
            ):

                boosted_scores[position] += 0.08


        # Disadvantages
        elif intent == "disadvantages":

            if (
                "disadvantage" in kb_question
                or "disadvantages" in kb_question
            ):

                boosted_scores[position] += 0.08


        # Uses
        elif intent == "uses":

            if (
                "use" in kb_question
                or "uses" in kb_question
                or "application" in kb_question
                or "applications" in kb_question
            ):

                boosted_scores[position] += 0.08


    # --------------------------------------------------------
    # FIND BEST MATCH
    # --------------------------------------------------------

    best_position = int(
        np.argmax(
            boosted_scores
        )
    )

    best_index = int(
        candidate_indices[
            best_position
        ]
    )


    # --------------------------------------------------------
    # ORIGINAL SEMANTIC SCORE
    # --------------------------------------------------------

    best_score = float(
        similarity_scores[
            best_index
        ]
    )


    # --------------------------------------------------------
    # MATCHED QUESTION / ANSWER
    # --------------------------------------------------------

    best_question = (
        df.iloc[
            best_index
        ]["question"]
    )

    best_answer = (
        df.iloc[
            best_index
        ]["answer"]
    )


    # --------------------------------------------------------
    # THRESHOLD
    # --------------------------------------------------------

    if best_score >= threshold:

        return {

            "answer": best_answer,

            "matched_question":
                best_question,

            "score":
                best_score,

            "matched":
                True,

            "topic":
                topic,

            "is_follow_up":
                is_follow_up_question,

            "intent":
                intent,

            "search_query":
                search_query
        }


    # --------------------------------------------------------
    # FALLBACK
    # --------------------------------------------------------

    return {

        "answer": (
            "Sorry, I couldn't find a relevant "
            "answer in my educational knowledge base."
        ),

        "matched_question":
            best_question,

        "score":
            best_score,

        "matched":
            False,

        "topic":
            topic,

        "is_follow_up":
            is_follow_up_question,

        "intent":
            intent,

        "search_query":
            search_query
    }

## Context Analyzer

In [86]:
# ============================================================
# TEST CONTEXT-AWARE CHATBOT
# ============================================================

# Reset conversation topic
context_analyzer.active_topic = None


# Question 1
result1 = get_answer(
    "What is Python?"
)

print("USER:")
print("What is Python?")

print("\nBOT:")
print(result1["answer"])

print("\nTopic:", result1["topic"])
print("Follow-up:", result1["is_follow_up"])
print("Search Query:", result1["search_query"])


# Question 2
result2 = get_answer(
    "What is its type?"
)

print("\n\nUSER:")
print("What is its type?")

print("\nBOT:")
print(result2["answer"])

print("\nTopic:", result2["topic"])
print("Follow-up:", result2["is_follow_up"])
print("Intent:", result2["intent"])
print("Search Query:", result2["search_query"])


# Question 3 - NEW TOPIC
result3 = get_answer(
    "What is SQL?"
)

print("\n\nUSER:")
print("What is SQL?")

print("\nBOT:")
print(result3["answer"])

print("\nTopic:", result3["topic"])
print("Follow-up:", result3["is_follow_up"])
print("Search Query:", result3["search_query"])


# Question 4 - SQL follow-up
result4 = get_answer(
    "What is its type?"
)

print("\n\nUSER:")
print("What is its type?")

print("\nBOT:")
print(result4["answer"])

print("\nTopic:", result4["topic"])
print("Follow-up:", result4["is_follow_up"])
print("Intent:", result4["intent"])
print("Search Query:", result4["search_query"])

USER:
What is Python?

BOT:
Python is a high-level, general-purpose programming language known for readable syntax and a large ecosystem of libraries.

Topic: Python
Follow-up: False
Search Query: What is Python?


USER:
What is its type?

BOT:
There are two types of data types in Python  1> Primtive (Integer, Float, Boolean, String, None)   2> Collective (List, Tuple, Set, Dictionary)

Topic: Python
Follow-up: True
Intent: types
Search Query: Python data types What is its type?


USER:
What is SQL?

BOT:
SQL, or Structured Query Language, is used to create, query, update, and manage data in relational database systems.

Topic: SQL
Follow-up: False
Search Query: What is SQL?


USER:
What is its type?

BOT:
There are so many data types in SQL 1> Numeric (Integer, Decimal, Float, Tiny Int, Big INT) 2> Binary (BLOB) 3> String (Char, Varchar) 4> Boolean 5> Date/Time.

Topic: SQL
Follow-up: True
Intent: types
Search Query: SQL data types What is its type?


## Semantic Analysis


In [88]:
def semantic_search(user_question, threshold=0.55):

    query_embedding = model.encode(
        user_question,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        [query_embedding],
        question_embeddings
    )[0]

    best_index = np.argmax(scores)
    best_score = scores[best_index]

    if best_score >= threshold:
        return (
            df.iloc[best_index]["answer"],
            best_score,
            df.iloc[best_index]["question"]
        )

    return None, best_score, None
answer, score, matched_question = semantic_search(
    "Do you understand Python?"
)

print("Answer:", answer)
print("Score:", round(score, 3))
print("Matched Question:", matched_question)

Answer: Python is a versatile and user-friendly programming language favored by data scientists and AI researchers for its simplicity and flexibility. It offers a vast ecosystem of libraries and frameworks tailored for machine learning, deep learning, and data analysis tasks. Python's readability and extensive community support make it an ideal choice for developing AI applications and conducting data-driven research across various domains.
Score: 0.773
Matched Question: Provide a short description of Python.


### Fallback Question

In [89]:
def fall_back_question(user_question, threshold=0.55):

    query_embedding = model.encode(
        user_question,
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        [query_embedding],
        question_embeddings
    )[0]

    best_index = np.argmax(scores)
    best_score = scores[best_index]

    if best_score >= threshold:
        return df.iloc[best_index]["answer"]

    return "Sorry, I do not have information related to this question."

question = "What is current PM of India?"

answer = fall_back_question(question)

print("\nAnswer:", answer)


Answer: Sorry, I do not have information related to this question.


## Hence , I have completed the project.